In [1]:
import pandas as pd
import numpy as np

In [2]:
df_od = pd.read_csv("/Users/jmahn/ciel/DATA/publicDB/dongjak_OD_dateOder.csv")
df_staion = pd.read_csv("/Users/jmahn/ciel/DATA/publicDB/dongjak_stationID_lat_lon.csv")

In [3]:

# df_od["승차일시"] = pd.to_datetime(df_od["승차일시"], format="%Y%m%d%H%M%S", errors="coerce")
# df_od["하차일시"] = pd.to_datetime(df_od["하차일시"], format="%Y%m%d%H%M%S", errors="coerce")
df = df_od.sort_values(by="승차일시").reset_index(drop=True)
df.head()
# 

,카드번호,운행출발일시,트랜잭선ID,교통수단CD,환승횟수,버스노선ID,교통사업자ID,차량ID,사용자구분코드,승차일시,승차정류장ID,하차일시,하차정류장ID,이용객수_다인승
0,U0000080222177,20240531055303,23,105,1,11110602,111520020,111753597,1,20240601000004,9009721,20240601001538,9009660,1
1,U0000010505773,20240531043808,22,105,1,11110602,111520020,111753594,1,20240601000006,9010307,20240601000225,9009701,1
2,U0000006068553,20240531043808,99,105,0,11110602,111520020,111753594,1,20240601000009,9010307,20240601000618,9009684,1
3,U0000010067945,20240531053102,2,105,1,11110602,111520020,111758838,1,20240601000011,9009674,20240601000303,9009658,1
4,U0000012609491,20240531043808,82,105,1,11110602,111520020,111753594,1,20240601000011,9010307,20240601000155,9009701,1


In [9]:
type(df.iloc[56]["하차일시"])

str

In [4]:
# 정류장 ID 기준으로 -> 정류장 번호, 정류장명, 조정위도, 조정경도, 방위 넣기 
results=[]
for i, (pickID, dropID) in enumerate(zip(df["승차정류장ID"], df["하차정류장ID"])):
    try:
        pick_station_row = df_staion[df_staion["ID"] == int(pickID)]
    except ValueError:
        pick_station_row = pd.DataFrame()  
        pickID=None
    
    try:
        drop_station_row = df_staion[df_staion["ID"] == int(dropID)]
    except ValueError:
        drop_station_row = pd.DataFrame()  
        dropID=None
    
    od_row = df.iloc[i]
    # 기본값 설정 (정류장 정보가 없을 경우 대비)
    pick_num = pick_name = pick_lat = pick_lon = pick_azimuth = None
    drop_num = drop_name = drop_lat = drop_lon = drop_azimuth = None 
    # bus_departure_time = pick_time = None
    # drop_time = None
    if not pick_station_row.empty:
        pick_num = int(pick_station_row.iloc[0]["정류장번호"])
        pick_name = pick_station_row.iloc[0]["정류장명"]
        pick_lat = pick_station_row.iloc[0]["조정위도"]
        pick_lon = pick_station_row.iloc[0]["조정경도"]
        pick_azimuth = int(pick_station_row.iloc[0]["방위"])
        
    if not drop_station_row.empty:
        drop_num = int(drop_station_row.iloc[0]["정류장번호"])
        drop_name = drop_station_row.iloc[0]["정류장명"]
        drop_lat = drop_station_row.iloc[0]["조정위도"]
        drop_lon = drop_station_row.iloc[0]["조정경도"]
        drop_azimuth = int(drop_station_row.iloc[0]["방위"])
    
    results.append({"운행출발일시": od_row["운행출발일시"], 
                    "승차일시": od_row["승차일시"], 
                    "승차정류장ID": pickID,
                    "승차정류장번호": pick_num, 
                    "승차정류장명": pick_name, 
                    "승차정류장위도": pick_lat, 
                    "승차정류장경도": pick_lon, 
                    "승차정류장방위": pick_azimuth,
                    "하차일시": od_row["하차일시"] if od_row["하차일시"] != '~' else None, 
                    "하차정류장ID": dropID, 
                    "하차정류장번호": drop_num, 
                    "하차정류장명": drop_name, 
                    "하차정류장위도": drop_lat, 
                    "하차정류장경도": drop_lon, 
                    "하차정류장방위": drop_azimuth,
                    "이용객수" : od_row["이용객수_다인승"], 
                    "환승횟수": od_row["환승횟수"]
    })
    # print(pickID, station_num, station_name, station_lat, station_lon, station_azimuth)
    # break
result_df = pd.DataFrame(results)
print(result_df.head())

           운행출발일시            승차일시  승차정류장ID      승차정류장번호   승차정류장명    승차정류장위도  \
0  20240531055303  20240601000004  9009721  119900126.0     노량진역  37.513504   
1  20240531043808  20240601000006  9010307  119900227.0  노량진신한은행  37.512155   
2  20240531043808  20240601000009  9010307  119900227.0  노량진신한은행  37.512155   
3  20240531053102  20240601000011  9009674  119900100.0    중앙대병원  37.507541   
4  20240531043808  20240601000011  9010307  119900227.0  노량진신한은행  37.512155   

      승차정류장경도  승차정류장방위            하차일시  하차정류장ID      하차정류장번호           하차정류장명  \
0  126.943248     87.0  20240601001538  9009660  119900092.0           은로초등학교   
1  126.944171    181.0  20240601000225  9009701  119900117.0            우성아파트   
2  126.944171    181.0  20240601000618  9009684  119900108.0              상도역   
3  126.961376    110.0  20240601000303  9009658  119900091.0  흑석한강센트레빌2차.흑석자이   
4  126.944171    181.0  20240601000155  9009701  119900117.0            우성아파트   

     하차정류장위도     하차정류장경도  하차정류장방위  이용객

In [5]:
result_df.to_csv("/Users/jmahn/ciel/DATA/publicDB/dongjak_OD_station_lat_lon.csv", index=False)
print("save finish!")

save finish!


In [9]:
result_df.head()

,운행출발일시,승차일시,승차정류장ID,승차정류장번호,승차정류장명,승차정류장위도,승차정류장경도,승차정류장방위,하차일시,하차정류장ID,하차정류장번호,하차정류장명,하차정류장위도,하차정류장경도,하차정류장방위,이용객수,환승횟수
0,20240531055303,20240601000004,9009721,119900126.0,노량진역,37.513504,126.943248,87.0,20240601001538,9009660,119900092.0,은로초등학교,37.503257,126.960781,198.0,1,1
1,20240531043808,20240601000006,9010307,119900227.0,노량진신한은행,37.512155,126.944171,181.0,20240601000225,9009701,119900117.0,우성아파트,37.508537,126.946766,138.0,1,1
2,20240531043808,20240601000009,9010307,119900227.0,노량진신한은행,37.512155,126.944171,181.0,20240601000618,9009684,119900108.0,상도역,37.503811,126.948998,57.0,1,0
3,20240531053102,20240601000011,9009674,119900100.0,중앙대병원,37.507541,126.961376,110.0,20240601000303,9009658,119900091.0,흑석한강센트레빌2차.흑석자이,37.501981,126.960221,200.0,1,1
4,20240531043808,20240601000011,9010307,119900227.0,노량진신한은행,37.512155,126.944171,181.0,20240601000155,9009701,119900117.0,우성아파트,37.508537,126.946766,138.0,1,1
